> **Weather-aware workflow note:** Weather observations are aligned to each generation timestamp. Forecast features use weather measured at forecast time t, never weather observed at the future target time t + 15 minutes.

In [1]:

import pandas as pd
import numpy as np
import os

In [2]:
plant1_gen = pd.read_csv("../data/raw/Plant_1_Generation_Data.csv")
plant1_weather = pd.read_csv("../data/raw/Plant_1_Weather_Sensor_Data.csv")

plant2_gen = pd.read_csv("../data/raw/Plant_2_Generation_Data.csv")
plant2_weather = pd.read_csv("../data/raw/Plant_2_Weather_Sensor_Data.csv")

In [3]:
print("Plant 1 Generation:", plant1_gen.shape)
print("Plant 1 Weather:", plant1_weather.shape)
print("Plant 2 Generation:", plant2_gen.shape)
print("Plant 2 Weather:", plant2_weather.shape)

Plant 1 Generation: (68778, 7)
Plant 1 Weather: (3182, 6)
Plant 2 Generation: (67698, 7)
Plant 2 Weather: (3259, 6)


In [4]:
print("Plant 1 Generation columns:")
print(plant1_gen.columns.tolist())

print("\nPlant 1 Weather columns:")
print(plant1_weather.columns.tolist())

print("\nPlant 2 Generation columns:")
print(plant2_gen.columns.tolist())

print("\nPlant 2 Weather columns:")
print(plant2_weather.columns.tolist())

Plant 1 Generation columns:
['DATE_TIME', 'PLANT_ID', 'SOURCE_KEY', 'DC_POWER', 'AC_POWER', 'DAILY_YIELD', 'TOTAL_YIELD']

Plant 1 Weather columns:
['DATE_TIME', 'PLANT_ID', 'SOURCE_KEY', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']

Plant 2 Generation columns:
['DATE_TIME', 'PLANT_ID', 'SOURCE_KEY', 'DC_POWER', 'AC_POWER', 'DAILY_YIELD', 'TOTAL_YIELD']

Plant 2 Weather columns:
['DATE_TIME', 'PLANT_ID', 'SOURCE_KEY', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']


In [5]:
# Convert DATE_TIME columns to datetime
plant1_gen["DATE_TIME"] = pd.to_datetime(
    plant1_gen["DATE_TIME"], dayfirst=True
)

plant1_weather["DATE_TIME"] = pd.to_datetime(
    plant1_weather["DATE_TIME"], dayfirst=True
)

plant2_gen["DATE_TIME"] = pd.to_datetime(
    plant2_gen["DATE_TIME"], dayfirst=True
)

plant2_weather["DATE_TIME"] = pd.to_datetime(
    plant2_weather["DATE_TIME"], dayfirst=True
)

# Check data types
print("Plant 1 Generation:", plant1_gen["DATE_TIME"].dtype)
print("Plant 1 Weather:", plant1_weather["DATE_TIME"].dtype)
print("Plant 2 Generation:", plant2_gen["DATE_TIME"].dtype)
print("Plant 2 Weather:", plant2_weather["DATE_TIME"].dtype)

Plant 1 Generation: datetime64[us]
Plant 1 Weather: datetime64[us]
Plant 2 Generation: datetime64[us]
Plant 2 Weather: datetime64[us]


C:\Users\ritik\AppData\Local\Temp\ipykernel_16920\3044697637.py:6: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  plant1_weather["DATE_TIME"] = pd.to_datetime(
C:\Users\ritik\AppData\Local\Temp\ipykernel_16920\3044697637.py:10: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  plant2_gen["DATE_TIME"] = pd.to_datetime(
C:\Users\ritik\AppData\Local\Temp\ipykernel_16920\3044697637.py:14: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  plant2_weather["DATE_TIME"] = pd.to_datetime(


In [6]:
print("Plant 1 Generation:")
print("Start:", plant1_gen["DATE_TIME"].min())
print("End  :", plant1_gen["DATE_TIME"].max())

print("\nPlant 1 Weather:")
print("Start:", plant1_weather["DATE_TIME"].min())
print("End  :", plant1_weather["DATE_TIME"].max())

print("\nPlant 2 Generation:")
print("Start:", plant2_gen["DATE_TIME"].min())
print("End  :", plant2_gen["DATE_TIME"].max())

print("\nPlant 2 Weather:")
print("Start:", plant2_weather["DATE_TIME"].min())
print("End  :", plant2_weather["DATE_TIME"].max())

Plant 1 Generation:
Start: 2020-05-15 00:00:00
End  : 2020-06-17 23:45:00

Plant 1 Weather:
Start: 2020-05-15 00:00:00
End  : 2020-06-17 23:45:00

Plant 2 Generation:
Start: 2020-05-15 00:00:00
End  : 2020-06-17 23:45:00

Plant 2 Weather:
Start: 2020-05-15 00:00:00
End  : 2020-06-17 23:45:00


In [7]:
# Check how many unique timestamps are present
print("Plant 1 Generation timestamps:", plant1_gen["DATE_TIME"].nunique())
print("Plant 1 Weather timestamps:", plant1_weather["DATE_TIME"].nunique())

print("\nPlant 2 Generation timestamps:", plant2_gen["DATE_TIME"].nunique())
print("Plant 2 Weather timestamps:", plant2_weather["DATE_TIME"].nunique())

Plant 1 Generation timestamps: 3158
Plant 1 Weather timestamps: 3182

Plant 2 Generation timestamps: 3259
Plant 2 Weather timestamps: 3259


In [8]:
p1_missing = ~plant1_weather["DATE_TIME"].isin(
    plant1_gen["DATE_TIME"]
)

p2_missing = ~plant2_weather["DATE_TIME"].isin(
    plant2_gen["DATE_TIME"]
)

print("Plant 1 unmatched weather timestamps:", p1_missing.sum())
print("Plant 2 unmatched weather timestamps:", p2_missing.sum())

Plant 1 unmatched weather timestamps: 25
Plant 2 unmatched weather timestamps: 0


In [9]:
unmatched_p1 = plant1_weather.loc[
    p1_missing, ["DATE_TIME", "PLANT_ID", "SOURCE_KEY"]
]

print(unmatched_p1.to_string(index=False))

          DATE_TIME  PLANT_ID      SOURCE_KEY
2020-05-20 23:00:00   4135001 HmiyD2TTLFNqkNe
2020-05-20 23:15:00   4135001 HmiyD2TTLFNqkNe
2020-05-20 23:30:00   4135001 HmiyD2TTLFNqkNe
2020-05-20 23:45:00   4135001 HmiyD2TTLFNqkNe
2020-05-21 00:00:00   4135001 HmiyD2TTLFNqkNe
2020-05-21 00:15:00   4135001 HmiyD2TTLFNqkNe
2020-05-21 00:30:00   4135001 HmiyD2TTLFNqkNe
2020-05-23 05:15:00   4135001 HmiyD2TTLFNqkNe
2020-05-25 05:45:00   4135001 HmiyD2TTLFNqkNe
2020-05-28 22:30:00   4135001 HmiyD2TTLFNqkNe
2020-05-28 22:45:00   4135001 HmiyD2TTLFNqkNe
2020-05-28 23:00:00   4135001 HmiyD2TTLFNqkNe
2020-05-28 23:15:00   4135001 HmiyD2TTLFNqkNe
2020-05-28 23:30:00   4135001 HmiyD2TTLFNqkNe
2020-05-28 23:45:00   4135001 HmiyD2TTLFNqkNe
2020-05-29 00:00:00   4135001 HmiyD2TTLFNqkNe
2020-05-29 00:15:00   4135001 HmiyD2TTLFNqkNe
2020-05-29 00:30:00   4135001 HmiyD2TTLFNqkNe
2020-05-29 00:45:00   4135001 HmiyD2TTLFNqkNe
2020-05-29 01:00:00   4135001 HmiyD2TTLFNqkNe
2020-05-29 01:15:00   4135001 Hmiy

In [10]:
print("\nTime differences between unmatched timestamps:")

print(
    unmatched_p1["DATE_TIME"]
    .sort_values()
    .diff()
    .value_counts()
)


Time differences between unmatched timestamps:
DATE_TIME
0 days 00:15:00     20
2 days 04:45:00      1
2 days 00:30:00      1
3 days 16:45:00      1
19 days 04:30:00     1
Name: count, dtype: int64


In [11]:
plant1_merged = pd.merge(
    plant1_gen,
    plant1_weather,
    on=["DATE_TIME", "PLANT_ID"],
    how="inner",
    suffixes=("_gen", "_weather")
)

print("Plant 1 merged shape:", plant1_merged.shape)
print("\nColumns:")
print(plant1_merged.columns.tolist())

Plant 1 merged shape: (68774, 11)

Columns:
['DATE_TIME', 'PLANT_ID', 'SOURCE_KEY_gen', 'DC_POWER', 'AC_POWER', 'DAILY_YIELD', 'TOTAL_YIELD', 'SOURCE_KEY_weather', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']


In [12]:
plant2_merged = pd.merge(
    plant2_gen,
    plant2_weather,
    on=["DATE_TIME", "PLANT_ID"],
    how="inner",
    suffixes=("_gen", "_weather")
)

print("Plant 2 merged shape:", plant2_merged.shape)
print("\nColumns:")
print(plant2_merged.columns.tolist())

Plant 2 merged shape: (67698, 11)

Columns:
['DATE_TIME', 'PLANT_ID', 'SOURCE_KEY_gen', 'DC_POWER', 'AC_POWER', 'DAILY_YIELD', 'TOTAL_YIELD', 'SOURCE_KEY_weather', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']


In [13]:
print("Plant 1 SOURCE_KEY mismatches:")

p1_source_mismatch = (
    plant1_merged["SOURCE_KEY_gen"]
    != plant1_merged["SOURCE_KEY_weather"]
)

print(p1_source_mismatch.sum())


print("\nPlant 2 SOURCE_KEY mismatches:")

p2_source_mismatch = (
    plant2_merged["SOURCE_KEY_gen"]
    != plant2_merged["SOURCE_KEY_weather"]
)

print(p2_source_mismatch.sum())

Plant 1 SOURCE_KEY mismatches:
68774

Plant 2 SOURCE_KEY mismatches:
67698


In [14]:
# Remove duplicate weather SOURCE_KEY
plant1_merged = plant1_merged.drop(columns=["SOURCE_KEY_weather"])
plant2_merged = plant2_merged.drop(columns=["SOURCE_KEY_weather"])

# Rename generation SOURCE_KEY to simply SOURCE_KEY
plant1_merged = plant1_merged.rename(
    columns={"SOURCE_KEY_gen": "SOURCE_KEY"}
)

plant2_merged = plant2_merged.rename(
    columns={"SOURCE_KEY_gen": "SOURCE_KEY"}
)

print("Plant 1 columns:")
print(plant1_merged.columns.tolist())

print("\nPlant 2 columns:")
print(plant2_merged.columns.tolist())

Plant 1 columns:
['DATE_TIME', 'PLANT_ID', 'SOURCE_KEY', 'DC_POWER', 'AC_POWER', 'DAILY_YIELD', 'TOTAL_YIELD', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']

Plant 2 columns:
['DATE_TIME', 'PLANT_ID', 'SOURCE_KEY', 'DC_POWER', 'AC_POWER', 'DAILY_YIELD', 'TOTAL_YIELD', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']


In [15]:
print("Plant 1 missing values:")
print(plant1_merged.isnull().sum())

print("\nPlant 2 missing values:")
print(plant2_merged.isnull().sum())

Plant 1 missing values:
DATE_TIME              0
PLANT_ID               0
SOURCE_KEY             0
DC_POWER               0
AC_POWER               0
DAILY_YIELD            0
TOTAL_YIELD            0
AMBIENT_TEMPERATURE    0
MODULE_TEMPERATURE     0
IRRADIATION            0
dtype: int64

Plant 2 missing values:
DATE_TIME              0
PLANT_ID               0
SOURCE_KEY             0
DC_POWER               0
AC_POWER               0
DAILY_YIELD            0
TOTAL_YIELD            0
AMBIENT_TEMPERATURE    0
MODULE_TEMPERATURE     0
IRRADIATION            0
dtype: int64


In [16]:
print("Plant 1 duplicate rows:", plant1_merged.duplicated().sum())
print("Plant 2 duplicate rows:", plant2_merged.duplicated().sum())

Plant 1 duplicate rows: 0
Plant 2 duplicate rows: 0


In [17]:
plant1_merged.to_csv(
    "../data/processed/plant1_merged.csv",
    index=False
)

plant2_merged.to_csv(
    "../data/processed/plant2_merged.csv",
    index=False
)

print("Merged datasets saved successfully.")

Merged datasets saved successfully.


In [18]:
import os

print("Plant 1 file exists:",
      os.path.exists("../data/processed/plant1_merged.csv"))

print("Plant 2 file exists:",
      os.path.exists("../data/processed/plant2_merged.csv"))

Plant 1 file exists: True
Plant 2 file exists: True
